# MoveLab 运动视觉 PBL
从像素到可复核的测量。以下数据均为合成，课程练习不引用 MoveLab 内部实现或运动会学生影像。
运行 `python -m pip install -r requirements.txt` 安装审阅版所用依赖，再在 Notebook 中按顺序执行各单元。

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 地面标定与跳远
四个角的输入为归一化图像坐标，输出为单位矩形。矩形长 3 米。示例以角点解八个方程，在角点之外用两个已知点检查误差。
实拍请人工确认最近触地痕迹，保留落点帧、标定角和卷尺参考值。飞行点不参与地面测距。

In [ ]:
def fit_plane(src, world):
    rows, rhs = [], []
    for (x,y), (u,v) in zip(src,world,strict=True):
        rows.extend([[x,y,1,0,0,0,-u*x,-u*y],[0,0,0,x,y,1,-v*x,-v*y]])
        rhs.extend([u,v])
    a=np.asarray(rows)
    if np.linalg.cond(a)>1e6:
        raise ValueError("unstable calibration")
    return np.r_[np.linalg.lstsq(a,rhs,rcond=None)[0],1].reshape(3,3)

def project(h,p):
    answer=h@np.r_[p,1]
    if abs(answer[2])<1e-9:
        raise ValueError("point near projection pole")
    return answer[:2]/answer[2]

def camera(x,y):
    return np.array([(.13+.74*x+.34*y)/(1+.70*y),(.88-.57*y)/(1+.70*y)])

world=[(0,0),(1,0),(1,1),(0,1)]
h=fit_plane([camera(*p) for p in world],world)
for truth in [(0.25,0.33),(0.72,0.67)]:
    mapped=project(h,camera(*truth))
    print("independent check error (cm)",round(100*np.linalg.norm((mapped-truth)*[1,3]),6))
    assert np.allclose(mapped,truth)
contacts=[camera(.27,.52),camera(.72,.57),camera(.48,.43)]
distances=[project(h,p)[1]*3 for p in contacts]
print("contacts in m", np.round(distances,2), "closest", round(min(distances),2))

## 过线与关节角
用带符号距离夹逼过线，线性插值只在前后观测均可用时计算。保留夹逼区间，以比较时钟 OCR 和源视频 PTS。
测关节角时使用等比例像素坐标。任何一个点不可见或两点重合时，输出未知。

In [ ]:
def crossing(t0,t1,d0,d1):
    if not (t0<t1 and d0<0<=d1):
        return None
    return t0 + (t1-t0)*(-d0)/(d1-d0)

def joint_angle(a,b,c,confidence):
    if min(confidence)<.55:
        return None
    u,v=np.subtract(a,b),np.subtract(c,b)
    norm=np.linalg.norm(u)*np.linalg.norm(v)
    return float(np.degrees(np.arccos(np.clip(u@v/norm,-1,1)))) if norm>1e-8 else None

assert crossing(820,920,-12,8)==880
assert crossing(820,920,-12,-2) is None
assert joint_angle([0,1],[0,0],[1,0],[.9,.9,.9])==90
assert joint_angle([0,1],[0,0],[1,0],[.9,.2,.9]) is None
print("crossing:",crossing(820,920,-12,8),"ms, bracket 820-920 ms")

## 训练事件分类器：跨场次留出
每一行是终点附近的时间窗口。先模拟事件类别，再按类别生成带重叠分布的特征；相同场次共享机位偏移。特征为三帧有符号距离（cm）、速度一致性和最低可见性。
按场次划分数据。低可见性样本在模型和规则路径中都进入未知。用验证集选 F1 最大的阈值，只在留出集评分。AP 指 average precision。

In [ ]:
rng=np.random.default_rng(20261126)
X,y,groups=[],[],[]
means=np.array([[-19,-7,8,.84],[-16,-10,-6,.65],[-14,2,-12,.46],[13,4,-8,.74]])
for session in range(36):
    offset=rng.normal(0,3)
    for kind in rng.choice(4,36,p=[.34,.3,.2,.16]):
        signals=means[kind]+rng.normal(0,[5,4,6,.19])
        signals[:3]+=offset
        visibility=rng.uniform(.28,1)
        if visibility<.55:
            signals[:3]+=rng.normal(0,7,3)
        X.append([*signals,visibility])
        y.append(int(kind==0))
        groups.append(session)
X,y,groups=map(np.asarray,(X,y,groups))
pool,test=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=7).split(X,y,groups))
a,b=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=11).split(X[pool],y[pool],groups[pool]))
train,valid=pool[a],pool[b]
assert len(set(groups[train])|set(groups[valid])|set(groups[test]))==36
assert not set(groups[train])&set(groups[valid])
assert not set(groups[train])&set(groups[test])
assert not set(groups[valid])&set(groups[test])
eligible=lambda idx: idx[X[idx,4]>=.55]
train_ok,valid_ok,test_ok=map(eligible,(train,valid,test))
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)).fit(X[train_ok],y[train_ok])
scores=lambda idx: clf.predict_proba(X[idx])[:,1]
validation=scores(valid_ok)
def metrics(truth,pred):
    return precision_recall_fscore_support(truth,pred,average='binary',zero_division=0)[:3]
gate=max(np.linspace(.15,.85,71),key=lambda z:metrics(y[valid_ok],validation>=z)[2])
proba=scores(test_ok)
baseline=(X[test_ok,1]<0)&(X[test_ok,2]>=0)
print("scenes train/valid/test:",*[len(set(groups[k])) for k in (train,valid,test)])
print("coverage",len(test_ok),"/",len(test),"unknown fraction",round(1-len(test_ok)/len(test),3))
print("locked threshold",round(gate,2),"AP on eligible",round(average_precision_score(y[test_ok],proba),3))
for name,pred in [('rule',baseline),('learned',proba>=gate)]:
    print(name,"P/R/F1 on eligible",np.round(metrics(y[test_ok],pred),3))

## 篮球录像实验 · 从临时轨迹到场地站位

Roboflow 先检测和跟踪球员，再结合球队分组、号码 OCR 和名册处理身份，最后把站位映射到球场。这一节从已导出的观测开始，用合成例子检查后半段。号码只作为候选，跨帧至少两票且无并列才输出；脚点被遮挡时保留空值。每段录像的跟踪编号只在该段内有效。

- [Roboflow 篮球分析原版 Notebook](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/basketball-ai-how-to-detect-track-and-identify-basketball-players.ipynb)
- [vision-demos 投篮分析](https://github.com/jeremyipark/vision-demos/tree/main/basketball)

换成自己的检测结果后，分别核对轨迹换人、号码误读、脚点遮挡和校正后的站位。单个球衣号码不能确认真实姓名；空中球点不能直接投影成真实球速。

In [5]:
from collections import Counter

# 四个图像角点对应一个示例球场，地面单位为米。
court_m = [(0, 0), (28, 0), (28, 15), (0, 15)]
court_h = fit_plane([camera(*p) for p in world], court_m)

# 每条观测包含画面帧时刻、球队候选、临时轨迹号、号码读数、可见脚点。
observed = [
    (0.00, "A", 7, "11", camera(.50, .30)),
    (0.04, "A", 7, "11", camera(.51, .30)),
    (0.08, "A", 7, "17", None),
    (0.00, "B", 3, "11", camera(.25, .60)),
    (0.04, "B", 3, "17", camera(.26, .60)),
]

def number_candidate(numbers):
    ordered = Counter(n for n in numbers if n).most_common(2)
    if not ordered:
        return None
    number, votes = ordered[0]
    if votes < 2 or (len(ordered) > 1 and votes == ordered[1][1]):
        return None
    return number

team_a = [item for item in observed if item[1:3] == ("A", 7)]
team_b = [item for item in observed if item[1:3] == ("B", 3)]
print("A / track 7: number candidate", number_candidate(row[3] for row in team_a))
print("B / track 3: number candidate", number_candidate(row[3] for row in team_b))
ground = [None if row[4] is None else np.round(project(court_h, row[4]), 2).tolist()
          for row in team_a]
print("A / track 7: ground points (m)", ground)
assert ground == [[14.0, 4.5], [14.28, 4.5], None]
assert number_candidate(row[3] for row in team_b) is None


A / track 7: number candidate 11
B / track 3: number candidate None
A / track 7: ground points (m) [[14.0, 4.5], [14.28, 4.5], None]


## 迁移到实拍
标注最小字段建议包括 video_id、camera_id、subject_id、frame_pts_ms、class、visibility、reviewer、source_sha256。
先冻结分组与真值协议，再用实拍替换候选特征。检测模型另标目标框和遮挡，姿态另标可见关节。检测器在工作站微调并导出后，要在目标浏览器端复测。
报告未知率、端到端召回、配对的时间误差和推理耗时。合成得分仅用于检验训练流程。